# Notebook 07 — Autonomous Eval + Reliability + Report

**The question**: given a task, can the agent produce a mergeable diff — and reliably?

Per (agent × task × seed):
1. Fresh sandbox at the pinned SHA.
2. Run the agent (skip nav-only tasks — those were graded in notebook 06).
3. Capture diff + tool-call trace + wall-clock + (custom agent only) tokens.
4. Apply judges: rubric review, tests, static checks, sequence-aware tool-call score.
5. Cleanup.

Reliability sub-study: 3 seeds × 3 hardest tasks per agent. Single seed for everything else.

## 1. Setup

In [ ]:
import sys, yaml, time
from pathlib import Path
sys.path.insert(0, '.')

from utils.workspace import create_workspace
from utils.runners import run_claude_code, run_kiro, run_user_agent
from utils.checks import run_tests, run_static_checks
from utils.reporting import (
    build_results_frame, per_agent_summary, per_task_summary,
    failure_modes, reliability_summary, efficiency_summary,
)
from validators.traces import score_trace
from pr_reviewer import review, Rubric

TASKS_FILE = Path('scaffolding/tasks/tasks.yaml').resolve()
GROUND_TRUTH_DIR = Path('scaffolding/ground_truth').resolve()

tasks_doc = yaml.safe_load(TASKS_FILE.read_text())
REPO_META = tasks_doc['repo']
TASKS = tasks_doc['tasks']
# Skip nav-only tasks here — those were graded as Q&A in notebook 06.
AUTO_TASKS = [t for t in TASKS if not t.get('nav_only')]
print(f'{len(AUTO_TASKS)} autonomous tasks  |  {len(TASKS)-len(AUTO_TASKS)} nav-only skipped')

In [ ]:
import shutil

AGENTS = []
if shutil.which('claude'):
    AGENTS.append('claude_code')
if shutil.which('kiro-cli'):
    AGENTS.append('kiro')
AGENTS.append('my_agent')   # alias used in dispatch below

MAX_TASKS = None              # None = all autonomous tasks
PER_TASK_TIMEOUT = 900

# Reliability sub-study: extra seeds on the hardest tasks only.
RELIABILITY_SEEDS = [42, 1337, 7]
RELIABILITY_TASK_IDS = [t['id'] for t in AUTO_TASKS if t.get('difficulty') == 'hard'][:3]

task_subset = AUTO_TASKS if MAX_TASKS is None else AUTO_TASKS[:MAX_TASKS]
tasks_by_id = {t['id']: t for t in TASKS}
n_runs = sum(
    len(RELIABILITY_SEEDS) if t['id'] in RELIABILITY_TASK_IDS else 1
    for t in task_subset
) * len(AGENTS)
print(f'{len(AGENTS)} agents × {len(task_subset)} tasks ({len(RELIABILITY_TASK_IDS)} reliability)')
print(f'Total runs: {n_runs}')

## 2. Run loop

In [ ]:
def run_agent(agent, task, workspace, seed):
    if agent == 'claude_code':
        return run_claude_code(task, workspace, timeout=PER_TASK_TIMEOUT, seed=seed)
    if agent == 'kiro':
        return run_kiro(task, workspace, timeout=PER_TASK_TIMEOUT, seed=seed)
    return run_user_agent(
        task, workspace,
        module='my_agent',
        tasks_file=TASKS_FILE,
        cwd=Path.cwd(),
        timeout=PER_TASK_TIMEOUT, seed=seed,
    )

def judge_review(diff, task_id):
    rubric_path = GROUND_TRUTH_DIR / f'{task_id}.md'
    rubric = Rubric.from_path(rubric_path) if rubric_path.exists() else None
    if not diff.strip():
        return None
    try:
        return review(diff, rubric=rubric)
    except Exception as e:
        print(f'    review error: {e}')
        return None

def seeds_for(task_id):
    return RELIABILITY_SEEDS if task_id in RELIABILITY_TASK_IDS else [0]

rows = []
for agent in AGENTS:
    for task in task_subset:
        for seed in seeds_for(task['id']):
            t0 = time.time()
            print(f'[{agent} / {task["id"]} / seed={seed}] start')
            ws = create_workspace(
                repo_url=REPO_META['url'], pinned_sha=REPO_META['pinned_sha'],
                agent=agent, task_id=task['id'],
            )
            try:
                out = run_agent(agent, task, ws, seed)
                rev = judge_review(out.diff, task['id'])
                tests = run_tests(ws.repo_path)
                static = run_static_checks(ws.repo_path)
                trace = score_trace(out.tool_trace, task.get('expected_tools', {}), agent, task['id'])
                row = {
                    'agent': agent, 'task_id': task['id'], 'seed': seed,
                    'difficulty': task.get('difficulty'),
                    'is_trap': bool(task.get('is_trap')),
                    'review_pass': bool(rev and rev.passed),
                    'review_dimensions': (
                        {d.name: d.verdict for d in rev.dimensions} if rev else {}),
                    'tests_pass': tests.passed,
                    'tests_failed': tests.violations,
                    'static_pass': static.passed,
                    'static_violations': static.violations,
                    'tools_pass': trace.overall_pass,
                    'sequence_pass': trace.sequence_pass,
                    'tools_required_hit': trace.required_hit,
                    'tools_required_missed': trace.required_missed,
                    'tools_forbidden_hit': trace.forbidden_hit,
                    'sequence_notes': trace.sequence_notes,
                    'elapsed_s': round(out.elapsed_s, 1),
                    'tool_call_count': trace.n_calls,
                    'diff_chars': len(out.diff),
                    'input_tokens': out.input_tokens,
                    'output_tokens': out.output_tokens,
                    'error': out.error,
                }
                # On trap tasks, the correct outcome is an empty (or near-empty)
                # diff. Force review_pass false if the agent emitted real changes.
                if task.get('is_trap') and len(out.diff.strip()) > 200:
                    row['review_pass'] = False
                rows.append(row)
                overall = 'PASS' if all([row['review_pass'], row['tests_pass'],
                                         row['static_pass'], row['tools_pass']]) else 'FAIL'
                print(f'  -> {overall}  review={row["review_pass"]} '
                      f'tests={row["tests_pass"]} static={row["static_pass"]} '
                      f'tools={row["tools_pass"]} seq={row["sequence_pass"]}  '
                      f'({time.time()-t0:.0f}s)')
            finally:
                ws.cleanup()

print(f'\nCollected {len(rows)} rows.')

## 3. Autonomous scorecard

In [ ]:
df = build_results_frame(rows)
df[['agent','task_id','seed','difficulty','review_pass','tests_pass',
    'static_pass','tools_pass','sequence_pass','overall_pass',
    'tool_call_count','elapsed_s']]

In [ ]:
per_agent_summary(df)

In [ ]:
per_task_summary(df)

## 4. Reliability

Per-(agent, task) pass-rate across seeds. Only the reliability sub-study tasks have multiple seeds, so this table is filtered to those.

In [ ]:
reliability_summary(df)

## 5. Efficiency

Wall-clock seconds per task — the only **uniform** efficiency signal across all 3 agents (Kiro can't be intercepted for token-level cost). Tokens are populated for the custom agent only.

**Don't read this as cost-per-task across agents.** Wall-clock conflates model throughput, cold starts, network latency, and tool overhead. Use it as a within-agent tuning signal.

In [ ]:
efficiency_summary(df)

## 6. Failure modes

In [ ]:
failure_modes(df)

## 7. Drill down on a specific run

In [ ]:
import json
INSPECT_AGENT = df['agent'].iloc[0] if not df.empty else None
INSPECT_TASK = df['task_id'].iloc[0] if not df.empty else None

if INSPECT_AGENT and INSPECT_TASK:
    row = df[(df.agent == INSPECT_AGENT) & (df.task_id == INSPECT_TASK)].iloc[0]
    print(f'{INSPECT_AGENT} / {INSPECT_TASK}  ->  overall={"PASS" if row.overall_pass else "FAIL"}')
    print(f'tool calls:  required_hit={row.tools_required_hit}  missed={row.tools_required_missed}  forbidden={row.tools_forbidden_hit}')
    print(f'sequence notes: {row.sequence_notes}')
    print(f'review dimensions:')
    print(json.dumps(row.review_dimensions, indent=2))

## 8. Combined two-axis scorecard

Pull the pair-programmer summary from notebook 06's run (you can re-run it inline if you want to keep everything in one place) and stack it next to the autonomous summary. This is the single artifact you'd hand to a team picking an agent.

In [ ]:
# Optional: rebuild the pair-programmer scorecard inline if you saved its rows.
# Otherwise refer to notebook 06's output and combine by hand for now.
auto = per_agent_summary(df)
print('# Autonomous axis\n')
auto

## What to do with the results

- **Pick an agent for autonomous use** based on `overall_rate` × reliability across seeds.
- **Pick an agent for pair-programming** based on the notebook 06 scorecard (precision/recall/MRR + answer accuracy).
- **Tighten rubrics** where the review signal looks suspicious. Re-run notebook 04 to recalibrate.
- **Improve your MCP server prompt/docs** if `sequence_pass` is low across agents — agents are reaching for the right tools but ignoring their results.
- **Drop the PR reviewer into CI** — same reviewer that judged this eval. See `pr_reviewer/README.md` and `.github-action-example/pr-review.yml`.

## Reuse on your own repo

1. Edit `scaffolding/tasks/tasks.yaml` top-level `repo` section with your URL + SHA.
2. Re-run notebooks 02, 03 (rubrics + gold), pointing Claude at your repo.
3. Notebooks 04 onward don't change.